# 1. Introduction
-   **Weights & Biases Link:** [Link to my W&B report](https://wandb.ai/janis-kneubuehler-hochschule-luzern/janis-kneubuehler-nlp-project1/reports/I-BA_NLP-F2601-PIQA-Project-One--VmlldzoxNjM4MzI0MQ?accessToken=v75fk8fg8uj10657kmkivjcfcgrco9imi4h12vbhq6nm4unch0l2am3y92v9p05h)
-   **Tools & Sources Used:**
    - AI Tools
        - NotebookLM: Summarizing course materials and project description, Creating Template of Python Notebook with titles and descriptions
        - Gemini: Coding support / Learning help
        - Claude: Coding support
    - Others
        - Deepl: Translation 
        - PyTorch documentation
        - Course documents
        - Medium articles (important ones are in the specific markdown cells)


# 2. Setup
I used libraries that were covered in the course. I didn't evaluate different libraries.<br>I chose the ones with the biggest community in web.
Seed: I've also added the seed to the random library, because i wasn't sure if it is used by any other libraries as a sublibrary. Just to be on the safe side.

In [ ]:
# !pip install datasets gensim wandb nltk torch

In [1]:
import torch
from datasets import load_dataset
import wandb
import nltk
import numpy as np
import random

/Users/janis.kneubuehler/workspace/NLP/NLP_Exercises/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 3. Preprocessing & Data Loading
**Decisions:**
- Tokenization: I use nltk.word_tokenize as recommended in the project discussion and demonstrated in the exercise of SW02.
- Lowercase: I convert to lowercase to reduce the vocabulary size, therefore we have less unknown words.
- Removal of punctuation: I remove punctuation for reducing vocabulary size and because they have no semantic meaning. with the nltk.word_tokenize they are separate tokens and easy to remove. 
- Removal of stopwords: I don't remove stopwerds, because default stopwords like "in", "on", etc. are critical for understanding physical relationships. With removal of them the project wouldn't make any sense.
- Stemming and Lemmatization: I skip stemming and lemmatizing because the word2vec is huge and has different word forms in it. Stemming would maybe perform worse, because some "not-real words" wouldn't be recognized by word2vec.
- Cleaning unknown words: I map unkknown words into an "unk" token. This was suggested by Gemini as it is a simple way to prevent KeyErros while training and it is a common method. (see "Decisions specific for Vocabulary")
- Format cleaning: No html removal necessary, because it is already cleaned in the dataset. (see "Data Exploration")
- Truncation: No truncation made, because it could cut off important words in the sentence and change the whole meaning. (e.g. I am on the house -REMOVED: after I climed up the ladder.-) This was discussed in a project discussion.
- Input format: I concatenate the goal with both solution and a separator, so from one row i get two: goal + "sep" + solution1 and goal + "sep" + solution2. The separator explicitly signals to the model where the context ends and the solution begins, this prevents the two sentences from blending together. (see "Decisions specific for Vocabulary")
- Label format: The original binary label indicates which solution is correct. This label is used directly as a class index for CrossEntropyLoss.
- Batching, padding: see "Decisions specific for DataLoader"
- Vocabulary, Embedding: see "Decisions specific for DataLoader"

In [3]:
# Load the PIQA dataset from revision because dataset scripts are no longer supported
# Use splits like specified in Course Project Slides
train = load_dataset('ybisk/piqa', split='train[:-1000]', revision='refs/convert/parquet')
valid = load_dataset('ybisk/piqa', split='train[-1000:]', revision='refs/convert/parquet')
test = load_dataset('ybisk/piqa', split='validation', revision='refs/convert/parquet')

In [4]:
import re

# Exploring the data
print(f"training: {len(train)}, validation: {len(valid)}, test: {len(test)}")
print("---------------------------")

print("Exploring some random rows in dataset:")
print(train[0])
print(train[12])
print(train[123])
print(train[4563])
print(train[13245]) 
print("---------------------------")

# Count rows with html (Regex was generated with Gemini and validated by me with https://regex101.com/)
goals_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['goal'], re.IGNORECASE))
sols1_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['sol1'], re.IGNORECASE))
sols2_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['sol2'], re.IGNORECASE))

print(f"Number of rows with html (for format cleaning): {len(goals_with_html)}, {len(sols1_with_html)}, {len(sols2_with_html)}")
print("---------------------------")
print(f"Class distribution in train: 0: {sum(1 for ex in train if ex['label']==0)} and 1: {sum(1 for ex in train if ex['label']==1)}")

training: 15113, validation: 1000, test: 1838
---------------------------
Exploring some random rows in dataset:
{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
{'goal': 'how do you open a capri-sun', 'sol1': 'open the straw attached to the juice, and then stick it in the small hole at the front of the pouch.', 'sol2': 'cut the top off with scissors', 'label': 0}
{'goal': 'How do you open a gift without ripping the paper?', 'sol1': 'Hold gift over boiling water to let the steam release the tape adhesive.', 'sol2': 'Hold gift over cold water to let the steam release the tape adhesive.', 'label': 0}
{'goal': 'how to make fancy piped flowers', 'sol1': 'use molded ganache', 'sol2': 'use Russian piping tips', 'label': 1}
{'goal': 'How do you keep steel wool and cotton dry?', 'sol1': 'Place the make-shift fire starter inside a film canister.', 'sol2': 'Place the make-shift fire starter inside your pocket.', '

In [ ]:
# IMPORTANT: This is a workaround on mac for SSL certificate error (Code by Gemini)
# I let this code in for my development and commented it out for the submission. This code should be ignored. 

# import os
# import ssl
# import certifi

# # 1. Point the notebook to your trusted certificates
# os.environ['SSL_CERT_FILE'] = certifi.where()
# os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# # 2. Force Python to allow unverified contexts globally (The safety net)
# try:
#     _create_unverified_https_context = ssl._create_unverified_context
# except AttributeError:
#     pass
# else:
#     ssl._create_default_https_context = _create_unverified_https_context

# print("SSL Bypass active!")

SSL Bypass active!


In [6]:
import string

def preprocess(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [token for token in tokens if token not in string.punctuation]

    return tokens

def preprocess_data(data):
    goal_tokens = preprocess(data['goal'])
    sol1_tokens = preprocess(data['sol1'])
    sol2_tokens = preprocess(data['sol2'])
    
    # concat goal with input
    data['input1_tokens'] = goal_tokens + ['<sep>'] + sol1_tokens
    data['input2_tokens'] = goal_tokens + ['<sep>'] + sol2_tokens
    
    return data

train = train.map(preprocess_data)
valid = valid.map(preprocess_data)
test = test.map(preprocess_data)

print(train[0])


{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1, 'input1_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<sep>', 'pour', 'it', 'onto', 'a', 'plate'], 'input2_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<sep>', 'pour', 'it', 'into', 'a', 'jar']}


## Decisions specific for Vocabulary:
- I only use training data for the vocabulary to prevent data leakage into validation or test
- It would be possible to do the vocabulary with torchtext. I didn't use torchtext because it was not part of our course material.
- I added special tokens for padding (pad), unknown words (unk) and separator for separating goal and solution (sep).
- I did not analyse the unknown words of test set to get not biased.
This code was built with help of Claude. The decisions made was evaluated with help of Gemini.

In [7]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {'<pad>': 0, '<unk>': 1, '<sep>': 2}
        self.idx2word = {0: '<pad>', 1: '<unk>', 2: '<sep>'}
        self.idx = 3

    def build_vocab(self, dataset):
        for example in dataset:
            for token in example['input1_tokens'] + example['input2_tokens']:
                if token not in self.word2idx:
                    self.word2idx[token] = self.idx
                    self.idx2word[self.idx] = token
                    self.idx += 1

    def encode(self, tokens):
        encoded = []
        unk_count = 0
        for token in tokens:
            if token in self.word2idx:
                encoded.append(self.word2idx[token])
            else:
                encoded.append(self.word2idx['<unk>'])
                unk_count += 1
        return encoded, unk_count

vocab = Vocabulary()
vocab.build_vocab(train)
print(f"Total unique words in Training Vocabulary: {vocab.idx}")

def encode_data(example):
    enc1, unk1 = vocab.encode(example['input1_tokens'])
    enc2, unk2 = vocab.encode(example['input2_tokens'])
    
    example['input1_ids'] = enc1
    example['input2_ids'] = enc2
    
    example['unk_count'] = unk1 + unk2
    return example

train = train.map(encode_data)
valid = valid.map(encode_data)
test = test.map(encode_data)

train_unks = sum(train['unk_count'])
valid_unks = sum(valid['unk_count'])
train_total = sum(len(ex['input1_ids']) + len(ex['input2_ids']) for ex in train)
valid_total  = sum(len(ex['input1_ids']) + len(ex['input2_ids']) for ex in valid)
print(f"<unk> tokens – Train: {train_unks} ({100*train_unks/train_total:.2f}%)")
print(f"<unk> tokens – Valid:  {valid_unks} ({100*valid_unks/valid_total:.2f}%)")


Total unique words in Training Vocabulary: 15766
<unk> tokens – Train: 0 (0.00%)
<unk> tokens – Valid:  781 (1.41%)


## Decisions specific for DataLoader:
- With pad_sequence I pad each batch to the longest sequence (dynamic padding). This was suggested by Gemini due to efficeny.
- I could easily train my Model 1 with a bigger batch size (up to 1024), but then i would only have 15 learning steps per epoch. This is why I use only batch size of 32 for the first trainings of Architecture One and increased to 64 for the next trainings of Architecture Two. <br>I fixed this hyperparameter at first. But after starting with Architecture Two I had to fix it again.
- I shuffle train data to avoid the model to memorize the order in the dataset.

In [8]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input1_list = []
    input2_list = []
    labels = []
    
    for example in batch:
        input1_list.append(torch.tensor(example['input1_ids']))
        input2_list.append(torch.tensor(example['input2_ids']))
        labels.append(example['label'])
        
    input1_padded = pad_sequence(input1_list, batch_first=True, padding_value=0)
    input2_padded = pad_sequence(input2_list, batch_first=True, padding_value=0)
    
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    
    return {
        'input1': input1_padded, 
        'input2': input2_padded, 
        'labels': labels_tensor
    }

batch_size = 64 

train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

first_batch = next(iter(train_loader))
print("Input 1 Batch Shape:", first_batch['input1'].shape)
print("Labels Batch Shape:", first_batch['labels'].shape)

Input 1 Batch Shape: torch.Size([64, 168])
Labels Batch Shape: torch.Size([64])


# 4. Model Definition
**Decisions:**
- **Pre-Trained Embedding:** I used word2vec because it was covered in the course exercises. FastText may be a little bit better for this use case, because it could handle better unknown words. (This was a good article for comparing those two: https://medium.com/swlh/a-quick-overview-of-the-main-difference-between-word2vec-and-fasttext-b9d3f6e274e9) I ruled out glove from the start because I didn't see any advantage over word2vec. In the end, I decided to go with word2vec because the course material is based on it.

- **Architecture 1:** I freeze the pre-trained embeddings so they don't get updated during training, as it was told in the project description. As written in the Project Discussions, a standard approach for input format is to average the word vectors. I implemented Mean Pooling for this, suggested by Gemini. By masking the "pad" tokens I get the correct average not biased by 0 vectors. The classifier is a 2-layer linear network (dimensions: 300, 128, 2) using a ReLU non-linearity and Dropout, which is tuned in Trainingprocess, for basic regularization.

- **Architecture 2:** Unlike Architecture 1 the embeddings here are unfrozen, to train the whole network end-to-end. I use a 2-layer bidirectional LSTM with 128 hidden dimension as required. The advantage of bidirectional is that in this use case reading a sentence right to left is at least as important as left to right to understand a physical situation. This behavior was explained to me by Gemini. To represent the entire sequence, I concatenate the final forward and backward hidden states to create a 256 dimensional vector. This vector is then scored by a 2-layer classifier (dimensions: 256, 128, 2). I used a standard ReLU non-linearity inside it to generate the final raw prediction logits.

In [ ]:
import gensim.downloader as api

# Load word2vec embeddings, used this tutorial: https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html 
wv = api.load('word2vec-google-news-300')
print(f"Vector size: {wv.vector_size}, Vocabulary size: {len(wv.index_to_key)}")

# Analyse word2vec
for index, word in enumerate(wv.index_to_key):
    if index == 10:
        break
    print(f"word #{index}/{len(wv.index_to_key)} is {word}")

Vector size: 300, Vocabulary size: 3000000
word #0/3000000 is </s>
word #1/3000000 is in
word #2/3000000 is for
word #3/3000000 is that
word #4/3000000 is is
word #5/3000000 is on
word #6/3000000 is ##
word #7/3000000 is The
word #8/3000000 is with
word #9/3000000 is said


In [ ]:
import torch.nn as nn

embedding_dim = wv.vector_size
vocab_size = len(vocab.word2idx)

embedding_matrix = torch.empty(vocab_size, embedding_dim)
nn.init.uniform_(embedding_matrix, -1.0 / embedding_dim, 1.0 / embedding_dim) # Claude helped here
embedding_matrix[vocab.word2idx['<pad>']] = torch.zeros(embedding_dim)

words_found = 0
for word, idx in vocab.word2idx.items():
    if word in wv:
        embedding_matrix[idx] = torch.from_numpy(np.array(wv[word], dtype=np.float32))
        words_found += 1

print(f"Filled {words_found} out of {vocab_size} words with pre-trained Word2Vec vectors. ({100*words_found/vocab_size:.1f}%) \nThe rest ist filled with random init.")

Filled 13348 out of 15766 words with pre-trained Word2Vec vectors. (84.7%) 
The rest ist filled with random init.


In [ ]:
class ArchitectureOne(nn.Module):
    
    def __init__(self, embedding_matrix, hidden_dim=128, dropout_rate=0.1):
        super().__init__()
        
        # Frozen pre-trained embeddings; padding_idx=0 zeroes out <pad> gradients (corrected by Claude)
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=True, padding_idx=0
        )
        
        embedding_dim = embedding_matrix.shape[1]
        
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, 1),
        )

    def _mean_pool(self, input_ids):
        embeds = self.embedding(input_ids)
        mask   = (input_ids != 0).float().unsqueeze(-1) # exclude padding
        summed = (embeds * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1e-9)
        return summed / lengths

    def forward(self, input1, input2):
        score1 = self.classifier(self._mean_pool(input1))
        score2 = self.classifier(self._mean_pool(input2))

        logits = torch.cat([score1, score2], dim=1)
        return logits

In [ ]:
class ArchitectureTwo(torch.nn.Module):
    def __init__(self, embedding_matrix, lstm_hidden_dim=128, classifier_hidden_dim=128, num_layers=2, dropout_rate=0.1):
        super().__init__()
        
        embedding_dim = embedding_matrix.shape[1]
        
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=False, padding_idx=0
        )
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_rate if num_layers > 1 else 0.0 # This change was made by Claude after Code Review before I didn't distinguish between num_layers
        )
        
        lstm_output_dim = 2 * lstm_hidden_dim
        
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(classifier_hidden_dim, 1)
        )
    
    # This Method was generated with Claude
    def _encode_sequence(self, input_ids):
        embeds = self.embedding(input_ids)
        _, (hn, _) = self.lstm(embeds)
        
        sentence_vec = torch.cat([hn[-2], hn[-1]], dim=-1) # Concatenate the final forward and backward hidden states of the top LSTM layer to represent the sequence.
        return sentence_vec
    
    def forward(self, input1, input2):
        score1 = self.classifier(self._encode_sequence(input1))
        score2 = self.classifier(self._encode_sequence(input2))

        logits = torch.cat([score1, score2], dim=1)
        return logits

# 5. Training
**Decissions / Clarification:**
- **Loss Function:** I use CrossEntropyLoss and not BCELoss. It is easier to understand for me and does not need a Sigmoid function. It should not make a difference in quality.
- **Optimizer & Regularization:** I use the Adam optimizer, which is a standard choice for these architectures and was suggested in the project discussion. I added a Weight Decay to the optimizer in Architecture 2, because it was overfitting. This was suggested by Gemini. In hindsight, I'd try the AdamW optimization for the architecture 2 as it is very high-dimensial with the unfrozen word2vec embeddings. (https://yassin01.medium.com/adam-vs-adamw-understanding-weight-decay-and-its-impact-on-model-performance-b7414f0af8a1)
- **Experiment Design:** My primary goal was to find a configuration that stops Architecture 2 from overfitting. To find this out, I kept the batch size locked at 64 and experimented with decreasing the learning rate (down to 1e-5) and after fixing lr increasing Dropout (up to 0.5) to force the model to generalize.
- **Number of training runs:** I ran approximately 10-15 manual training runs per model. I tracked all of them in Weights & Biases to compare the validation loss curves and identify the optimal hyperparameter combination for the final model.
- **Early Stopping & Checkpointing:** I implemented Early Stopping based on Validation Accuracy with a patience of 6 to 10 epochs. I save the last model if my rund would get aborted and the best model.
- **Validation:** I use torch.no_grad() during the validation and testing loops to disable gradient tracking. This was something Claude discovered that i missed. It guarantees the model doesn't learn from the validation set and saves memory.

In [ ]:
def train_model(model, train_loader, valid_loader, criterion, optimizer, checkpoint_name, epochs=20, patience=3):
    best_val_accuracy = 0.0
    patience_counter = 0
    checkpoint_best_path = f"best_{checkpoint_name}_model.pt"
    checkpoint_last_path = f"last_{checkpoint_name}_model.pt" # Save last model if my run gets aborted
    print(f"Number of batches {len(train_loader)}")

    for epoch in range(epochs):
        ########################## Training ##########################
        model.train()
        total_train_loss = 0.0
        
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['input1'], batch['input2'])
            loss   = criterion(logits, batch['labels'])
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        ########################## Validation ##########################
        model.eval()
        total_val_loss = 0.0
        correct_preds = 0
        total_preds = 0
        
        with torch.no_grad():
            for batch in valid_loader:
                logits = model(batch['input1'], batch['input2'])
                loss   = criterion(logits, batch['labels'])
                total_val_loss += loss.item()
                
                predictions = torch.argmax(logits, dim=1)
                correct_preds += (predictions == batch['labels']).sum().item()
                total_preds += batch['labels'].size(0)
                
        avg_val_loss = total_val_loss / len(valid_loader)
        val_accuracy = correct_preds / total_preds
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_accuracy
        })
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        ########################## Checkpoints ##########################
        torch.save(model.state_dict(), checkpoint_last_path)
        
        if val_accuracy > best_val_accuracy: # Best model & early stopping based on validation accuracy (as discussed in project discussion)
            print(f"Validation accuracy improved from {best_val_accuracy:.4f} to {val_accuracy:.4f}. Saving checkpoint.")
            best_val_accuracy = val_accuracy
            patience_counter = 0
            
            torch.save(model.state_dict(), checkpoint_best_path)
        else:
            patience_counter += 1
            print(f"No improvement in validation accuracy. Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs!")
                break # Exit the training loop

    print(f"Training complete. Saved best and last model.")

In [ ]:
import torch.optim as optim

# Login to W&B
wandb.login()

wandb.init(
    project="janis-kneubuehler-nlp-project1", 
    config={
        "architecture": "ArchitectureOne",
        "run_goal": "decrease dropout",
        "learning_rate": 1e-3,
        "hidden_dim": 128,
        "dropout": 0.2,
        "epochs": 30,
        "batch_size": batch_size
    }
)
config = wandb.config

model = ArchitectureOne(embedding_matrix, hidden_dim=128, dropout_rate=config.dropout)

criterion = nn.CrossEntropyLoss()

trainable_parameters = (p for p in model.parameters() if p.requires_grad)
optimizer = optim.Adam(trainable_parameters, lr=config.learning_rate)

train_model(model, train_loader, valid_loader, criterion, optimizer, checkpoint_name=f"arch1", epochs=config.epochs, patience=6)

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/janis.kneubuehler/.netrc.
wandb: Currently logged in as: janis-kneubuehler (janis-kneubuehler-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Number of batches 237
Epoch 1/30 | Train Loss: 0.6926 | Val Loss: 0.6918 | Val Acc: 0.5450
Validation accuracy improved from 0.0000 to 0.5450. Saving checkpoint.
Epoch 2/30 | Train Loss: 0.6901 | Val Loss: 0.6897 | Val Acc: 0.5500
Validation accuracy improved from 0.5450 to 0.5500. Saving checkpoint.
Epoch 3/30 | Train Loss: 0.6871 | Val Loss: 0.6880 | Val Acc: 0.5660
Validation accuracy improved from 0.5500 to 0.5660. Saving checkpoint.
Epoch 4/30 | Train Loss: 0.6847 | Val Loss: 0.6869 | Val Acc: 0.5580
No improvement in validation accuracy. Patience: 1/6
Epoch 5/30 | Train Loss: 0.6824 | Val Loss: 0.6861 | Val Acc: 0.5580
No improvement in validation accuracy. Patience: 2/6
Epoch 6/30 | Train Loss: 0.6791 | Val Loss: 0.6860 | Val Acc: 0.5530
No improvement in validation accuracy. Patience: 3/6
Epoch 7/30 | Train Loss: 0.6781 | Val Loss: 0.6854 | Val Acc: 0.5640
No improvement in validation accuracy. Patience: 4/6
Epoch 8/30 | Train Loss: 0.6743 | Val Loss: 0.6847 | Val Acc: 0.5590
N

epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
train_loss,██▇▇▆▆▆▅▄▄▄▃▃▂▂▁▁
val_accuracy,▁▂▆▄▄▃▆▅▆▆█▄▆▆▆▇▇
val_loss,█▆▅▄▄▄▃▃▂▂▁▂▁▁▁▁▁
epoch,17
train_loss,0.65118
val_accuracy,0.569
val_loss,0.68276


In [ ]:
import torch.optim as optim

# Login to W&B
wandb.login()

wandb.init(
    project="janis-kneubuehler-nlp-project1", 
    config={
        "architecture": "ArchitectureTwo",
        "run_goal": "decrease learning_rate due to model starvation",
        "learning_rate": 1e-4,
        'lstm_hidden_dim': 128,
        'classifier_hidden_dim': 128,
        'num_layers': 2,
        "dropout": 0.5,
        "epochs": 30,
        "batch_size": batch_size
    }
)
config2 = wandb.config

model2 = ArchitectureTwo(embedding_matrix, lstm_hidden_dim=config2.lstm_hidden_dim, classifier_hidden_dim=config2.classifier_hidden_dim, num_layers=config2.num_layers, dropout_rate=config2.dropout)
criterion2  = nn.CrossEntropyLoss()
optimizer2  = optim.Adam(model2.parameters(), lr=config2.learning_rate, weight_decay=1e-4)

train_model(model2, train_loader, valid_loader, criterion2, optimizer2, checkpoint_name=f"arch2", epochs=config2.epochs, patience=10)

wandb.finish()

Number of batches 237
Epoch 1/30 | Train Loss: 0.6932 | Val Loss: 0.6931 | Val Acc: 0.5580
Validation accuracy improved from 0.0000 to 0.5580. Saving checkpoint.
Epoch 2/30 | Train Loss: 0.6929 | Val Loss: 0.6928 | Val Acc: 0.5690
Validation accuracy improved from 0.5580 to 0.5690. Saving checkpoint.
Epoch 3/30 | Train Loss: 0.5059 | Val Loss: 0.8388 | Val Acc: 0.5640
No improvement in validation accuracy. Patience: 1/10
Epoch 4/30 | Train Loss: 0.3809 | Val Loss: 0.9914 | Val Acc: 0.5650
No improvement in validation accuracy. Patience: 2/10
Epoch 5/30 | Train Loss: 0.3236 | Val Loss: 1.0782 | Val Acc: 0.5540
No improvement in validation accuracy. Patience: 3/10
Epoch 6/30 | Train Loss: 0.2724 | Val Loss: 1.2273 | Val Acc: 0.5510
No improvement in validation accuracy. Patience: 4/10
Epoch 7/30 | Train Loss: 0.2401 | Val Loss: 1.1573 | Val Acc: 0.5470
No improvement in validation accuracy. Patience: 5/10
Epoch 8/30 | Train Loss: 0.2140 | Val Loss: 1.3153 | Val Acc: 0.5500
No improvement

epoch,▁▂▂▃▄▄▅▅▆▇▇█
train_loss,██▆▄▃▃▂▂▂▁▁▁
val_accuracy,▅█▆▇▃▂▁▂▂▄▄▅
val_loss,▁▁▂▃▄▅▄▅▆▆▇█
epoch,12
train_loss,0.15089
val_accuracy,0.558
val_loss,1.68876


# 6. Evaluation
The models evaluated werde renamed manually by myself.
For the evaluation I used the models with the following reasons:
- Architecture 1: "fearless-silence-19" selected for its low validation loss (0.683) after decreasing dropout to 0.2. It safely avoided the overfitting seen in runs with artificially higher peak accuracies.
- Architecture 2: "trim-star-21" selected because it achieved the highest validation accuracy (57.7%) among all Architecture 2 experiments after successfully increasing the batch size to 64.

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss    = 0.0
    correct_preds = 0
    total_preds   = 0

    with torch.no_grad():
        for batch in loader:
            logits = model(batch['input1'], batch['input2'])
            loss   = criterion(logits, batch['labels'])
            total_loss    += loss.item()
            predictions    = torch.argmax(logits, dim=1)
            correct_preds += (predictions == batch['labels']).sum().item()
            total_preds   += batch['labels'].size(0)

    return total_loss / len(loader), correct_preds / total_preds

########################## Architecture 1 ##########################
model.load_state_dict(torch.load("./best_arch1_model.pt")) # My best Model of Arch1

val_loss1,  val_acc1  = evaluate(model, valid_loader, criterion)
test_loss1, test_acc1 = evaluate(model, test_loader,  criterion)
print(f"Architecture 1 – Validation, Loss: {val_loss1:.4f}, Accuracy: {val_acc1:.4f}")
print(f"Architecture 1 – Test, Loss: {test_loss1:.4f}, Accuracy: {test_acc1:.4f}")

########################## Architecture 2 ##########################
model2.load_state_dict(torch.load("./best_arch2_model.pt")) # My best Model of Arch2

val_loss2,  val_acc2  = evaluate(model2, valid_loader, criterion2)
test_loss2, test_acc2 = evaluate(model2, test_loader,  criterion2)
print(f"Architecture 2 – Validation, Loss: {val_loss2:.4f}, Accuracy: {val_acc2:.4f}")
print(f"Architecture 2 – Test, Loss: {test_loss2:.4f}, Accuracy: {test_acc2:.4f}")

Architecture 1 – Validation, Loss: 0.6821, Accuracy: 0.5750
Architecture 1 – Test, Loss: 0.6848, Accuracy: 0.5620
Architecture 2 – Validation, Loss: 0.7025, Accuracy: 0.5780
Architecture 2 – Test, Loss: 0.6924, Accuracy: 0.5773


# 7. Conclusion & Interpretation
**Results Summary:**
Both architectures achieved very similar results on the unseen test set, hovering around 56-57% accuracy. Since there are only two possible answers per question, random guessing would be 50%. While both models learned something, they clearly struggled with the task. The results are not much better than a coin flip.

**Model Comparison:**
Architecture 1: This model was incredibly fast to train. Because the embeddings were frozen, it was highly resistant to overfitting. However, because Mean Pooling blends all words into a single average vector, it loses the word order. Since the model completely ignores the grammatical order of the words, it knows which objects are in the sentence, but has no idea how they interact with each other.

Architecture 2: In my expections this model should be much smarter because the bidirectional LSTM reads the sequence and preserves grammar. However, because I unfroze the embeddings, the model had millions of trainable parameters. The training process was much longer and I had the problem of overfitting. Even with different strategies like high dropout and Weight Decay, it failed to generalize better than the simpler baseline.

**Interpretation:**
The PIQA dataset tests physical commonsense. Word2Vec embeddings only know which words appear next to each other in Web articles. They don't actually understand physical properties. These final results clearly demonstrate that it is extremely difficult for basic text-only models to learn real-world physics purely from the statistical distribution of words.